## Project #2. Scheduling the NBA ##

The National Basketball Association (NBA) is planning their schedule for the 2025/2026 season. In the attached games.csv you can find a (ficticious) preliminary schedule Among other data, you find, for each match, the home team, the away team, and the date and location of the match. The goal of the project is to possibly improve the current draft of the schedule, under some of the constraints imposed by the current schedule.

The project is composed of 4 questions, all to be solved using Integer Programming, to be completed in this order:

1. Write codes that computes and prints, for each team i, the following information:

(a) all the dates when team i played home; (b) for each team j, the number of times team i played against team j at home; (c) for each team j, the number of times team i played against team j away (i.e., at j's home); (d) all the dates when team j played away.

2. Write an Integer Program (without objective function) whose feasible solutions are all the feasible schedules, where a schedule is feasible if, for each team i:

(e) i plays home (possibly, against a different team) exactly on the dates computed in (a) above; (f) plays away (possibly, against a different team) exactly on the dates computed in (d) above; (g) for each team j, i plays home against team j exactly the number of times computed in (b) above; (h) for each team j, i plays away against team j (i.e., at j's home) exactly the number of times computed in (c) above.

3. Compute a feasible schedule that satisfy the following additional constraint: no team should play three consecutive matches where the sum of the absolute values of the difference between the time zones of two consecutive matches is 4 or more; or conclude that no such schedule exists. For instance, if a team plays game 1,2,3 and:
- the time zone difference between the arena where game 1 is played and the arena where game 2 is played is 2;
- the time zone difference between the arena where game 2 is played and the arena where game 3 is played is 3;
Then the schedue is infeasible, since 3+2 = 5 $\geq$ 4.

4. **This part will not be graded, but we may ask you about it in the one-on-one discussion** Compute any improvement to the current schedule. For instance, using Google Maps, you could compute and store the locations of all arenas, compute a feasible schedule that minimizes the maximum distance traveled by a team, and compare this value with the one of the current schedule to show the improvement. You can also access the full 2023/2024 season schedule at https://www.basketball-reference.com/leagues/NBA_2024_games-october.html.

**Deliverables**
- ipynb file containing all the code
- pdf file explaining the models, the variable, and the constraints
- Schedules saved in a .csv or analagous file, presenting the list of matches organized as follows: (a) date (b) team playing home (c) team playing away (d)

In [ ]:
%pip install pandas numpy
%pip install gurobi  
%pip install matplotlib seaborn 

Note: you may need to restart the kernel to use updated packages.
ERROR: Could not find a version that satisfies the requirement gurobi (from versions: none)
ERROR: No matching distribution found for gurobi
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Part 1

In [46]:
import pandas as pd
from collections import defaultdict

# Load data
df = pd.read_csv("games.csv")

teams = sorted(set(df["Home"]).union(df["Visitor"]))

# Data structures
home_dates = defaultdict(list)              # (a)
away_dates = defaultdict(list)              # (d)

home_vs_counts = {t: defaultdict(int) for t in teams}   # (b)
away_vs_counts = {t: defaultdict(int) for t in teams}   # (c)

# Populate
for _, row in df.iterrows():
    home = row["Home"]
    away = row["Visitor"]
    date = row["Date"]

    # (a) home dates
    home_dates[home].append(date)

    # (d) away dates
    away_dates[away].append(date)

    # (b) count home matches vs opponent
    home_vs_counts[home][away] += 1

    # (c) count away matches vs opponent
    away_vs_counts[away][home] += 1

# -------------------------
# PRINT RESULTS
# -------------------------

for t in teams:
    print("="*70)
    print(f"TEAM: {t}")
    print("-"*70)

    print("Home Dates:")
    for d in sorted(home_dates[t]):
        print(f"  - {d}")

    print("\nHome vs Opponent Counts:")
    for opp, c in home_vs_counts[t].items():
        print(f"  vs {opp}: {c}")

    print("\nAway vs Opponent Counts:")
    for opp, c in away_vs_counts[t].items():
        print(f"  at {opp}: {c}")

    print("\nAway Dates:")
    for d in sorted(away_dates[t]):
        print(f"  - {d}")

    print("\n")

TEAM: Atlanta Hawks
----------------------------------------------------------------------
Home Dates:
  - Fri, Nov 07, 2025
  - Fri, Nov 28, 2025
  - Mon, Nov 03, 2025
  - Mon, Nov 17, 2025
  - Sat, Nov 15, 2025
  - Sat, Nov 29, 2025
  - Sun, Nov 23, 2025
  - Thu, Dec 25, 2025
  - Thu, Nov 27, 2025
  - Wed, Nov 19, 2025

Home vs Opponent Counts:
  vs Toronto Raptors: 1
  vs Golden State Warriors: 1
  vs Boston Celtics: 1
  vs Milwaukee Bucks: 1
  vs Miami Heat: 1
  vs New York Knicks: 1
  vs Los Angeles Lakers: 1
  vs Phoenix Suns: 1
  vs Dallas Mavericks: 1
  vs Chicago Bulls: 1

Away vs Opponent Counts:
  at Chicago Bulls: 1
  at Brooklyn Nets: 1
  at Denver Nuggets: 1
  at Houston Rockets: 1
  at Philadelphia 76ers: 1
  at Cleveland Cavaliers: 1

Away Dates:
  - Fri, Nov 21, 2025
  - Mon, Dec 01, 2025
  - Sat, Nov 01, 2025
  - Thu, Nov 13, 2025
  - Tue, Nov 11, 2025
  - Wed, Nov 05, 2025


TEAM: Boston Celtics
----------------------------------------------------------------------
H

## Part 2

In [48]:
import pandas as pd
from collections import defaultdict
import gurobipy as gp
from gurobipy import GRB

print("Loading data...")
df = pd.read_csv("games.csv")

# Sets
T = sorted(set(df["Home"]).union(df["Visitor"]))     # teams
D = sorted(df["Date"].unique())                      # dates

# Parameters
home_games = {(i, d): 0 for i in T for d in D}
away_games = {(i, d): 0 for i in T for d in D}
Hij = {(i, j): 0 for i in T for j in T if i != j}
Aij = {(i, j): 0 for i in T for j in T if i != j}

for _, row in df.iterrows():
    home = row["Home"]
    away = row["Visitor"]
    d    = row["Date"]

    home_games[(home, d)] = 1
    away_games[(away, d)] = 1

    if home != away:
        Hij[(home, away)] += 1      # home team i hosts j
        Aij[(away, home)] += 1      # away team i is away @ j

m = gp.Model("nba_feasible_schedules")

# Variables x[i,j,d]
x_keys = []
for i in T:
    for j in T:
        if i == j:
            continue
        for d in D:
            if home_games[(i, d)] == 1 and away_games[(j, d)] == 1:
                x_keys.append((i, j, d))

x = m.addVars(x_keys, vtype=GRB.BINARY, name="x")

# (e) home dates fixed
for i in T:
    for d in D:
        m.addConstr(
            gp.quicksum(
                x[i, j, d]
                for j in T
                if i != j and (i, j, d) in x
            ) == home_games[(i, d)],
            name=f"home_date_{i}_{d}"
        )

# (f) away dates fixed
for j in T:
    for d in D:
        m.addConstr(
            gp.quicksum(
                x[i, j, d]
                for i in T
                if i != j and (i, j, d) in x
            ) == away_games[(j, d)],
            name=f"away_date_{j}_{d}"
        )

# (g) pairwise home counts
for i in T:
    for j in T:
        if i == j:
            continue
        m.addConstr(
            gp.quicksum(
                x[i, j, d] for d in D if (i, j, d) in x
            ) == Hij[(i, j)],
            name=f"home_count_{i}_{j}"
        )

# (h) pairwise away counts
for i in T:
    for j in T:
        if i == j:
            continue
        m.addConstr(
            gp.quicksum(
                x[j, i, d] for d in D if (j, i, d) in x
            ) == Aij[(i, j)],
            name=f"away_count_{i}_{j}"
        )

m.setObjective(0, GRB.MINIMIZE)
m.optimize()

print("Model status:", m.status)
if m.status == GRB.INFEASIBLE:
    print("Model reported infeasible, computing IIS...")
    m.computeIIS()
    m.write("infeasible.ilp")
    print("Wrote IIS to infeasible.ilp")

Loading data...
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 992 rows, 1024 columns and 4096 nonzeros
Model fingerprint: 0xf74fb265
Variable types: 0 continuous, 1024 integer (1024 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 684 rows and 582 columns
Presolve time: 0.00s
Presolved: 308 rows, 442 columns, 1386 nonzeros
Variable types: 0 continuous, 442 integer (442 binary)
Found heuristic solution: objective 0.0000000

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.01 work units)
Thread count was 10 (of 10 available processors)

Solution count 1: 0 

Optimal solution found (tolerance 1.00e-04)
Best objective 0.000000000000e+00, best bound 0.000000000000e+00, gap 0.0000%
Model st

In [41]:
import pandas as pd
from collections import defaultdict
import gurobipy as gp
from gurobipy import GRB
import os

# ============================================================================
# PART 1: LOAD DATA
# ============================================================================

df = pd.read_csv("games.csv")

# Make a parsed date for ordering
df["Date_parsed"] = pd.to_datetime(df["Date"])
teams = sorted(set(df["Home"]).union(df["Visitor"]))

# Time zone map (0 = Pacific, 1 = Mountain, 2 = Central, 3 = Eastern)
tz_map = {
    "Atlanta Hawks": 3,
    "Boston Celtics": 3,
    "Brooklyn Nets": 3,
    "Chicago Bulls": 2,
    "Cleveland Cavaliers": 3,
    "Dallas Mavericks": 2,
    "Denver Nuggets": 1,
    "Golden State Warriors": 0,
    "Houston Rockets": 2,
    "Los Angeles Lakers": 0,
    "Miami Heat": 3,
    "Milwaukee Bucks": 2,
    "New York Knicks": 3,
    "Philadelphia 76ers": 3,
    "Phoenix Suns": 1,
    "Toronto Raptors": 3,
}

# ============================================================================
# PART 2: RECOMPUTE (a)–(d)
#   (home_dates, away_dates, home_vs_counts, away_vs_counts)
# ============================================================================

home_dates = defaultdict(list)   # (a)
away_dates = defaultdict(list)   # (d)

home_vs_counts = {t: defaultdict(int) for t in teams}  # (b)
away_vs_counts = {t: defaultdict(int) for t in teams}  # (c)

for _, row in df.iterrows():
    home = row["Home"]
    away = row["Visitor"]
    date = row["Date"]

    home_dates[home].append(date)
    away_dates[away].append(date)

    home_vs_counts[home][away] += 1
    away_vs_counts[away][home] += 1

# Deduplicate and sort dates per team
home_dates = {t: sorted(set(dates), key=lambda x: pd.to_datetime(x)) for t, dates in home_dates.items()}
away_dates = {t: sorted(set(dates), key=lambda x: pd.to_datetime(x)) for t, dates in away_dates.items()}

# All game dates per team (home or away), sorted
team_dates = {}
for t in teams:
    dates_t = set(home_dates.get(t, [])) | set(away_dates.get(t, []))
    team_dates[t] = sorted(dates_t, key=lambda x: pd.to_datetime(x))

# ============================================================================
# PART 3: BUILD BASE IP (from Q2)
#   Decision variables x[i,d,j] = 1 if on date d, team i is HOME vs team j (j away)
# ============================================================================

m = gp.Model("nba_schedule_with_tz_constraints")

# Create x variables only when:
# - i != j
# - d is a home date for i
# - d is an away date for j
x_keys = []
for i in teams:
    for j in teams:
        if i == j:
            continue
        common_dates = set(home_dates.get(i, [])).intersection(away_dates.get(j, []))
        for d in common_dates:
            x_keys.append((i, d, j))

x = m.addVars(x_keys, vtype=GRB.BINARY, name="x")

# (e) Exactly one home opponent per home date
for i in teams:
    for d in home_dates.get(i, []):
        m.addConstr(
            gp.quicksum(
                x[i, d, j] for j in teams if i != j and (i, d, j) in x
            ) == 1,
            name=f"home_slot_{i}_{d}"
        )

# (f) Exactly one away game per away date
for j in teams:
    for d in away_dates.get(j, []):
        m.addConstr(
            gp.quicksum(
                x[i, d, j] for i in teams if i != j and (i, d, j) in x
            ) == 1,
            name=f"away_slot_{j}_{d}"
        )

# (g) Home vs opponent counts
for i in teams:
    for j in teams:
        if i == j:
            continue
        target_home_ij = home_vs_counts[i][j]  # may be 0
        m.addConstr(
            gp.quicksum(
                x[i, d, j]
                for d in set(home_dates.get(i, [])).intersection(away_dates.get(j, []))
                if (i, d, j) in x
            ) == target_home_ij,
            name=f"home_vs_{i}_{j}"
        )

# (h) Away vs opponent counts
# away_vs_counts[i][j] = times i is away at j (so we look at x[j,d,i])
for i in teams:
    for j in teams:
        if i == j:
            continue
        target_away_ij = away_vs_counts[i][j]
        m.addConstr(
            gp.quicksum(
                x[j, d, i]
                for d in set(home_dates.get(j, [])).intersection(away_dates.get(i, []))
                if (j, d, i) in x
            ) == target_away_ij,
            name=f"away_vs_{i}_{j}"
        )

# ============================================================================
# PART 4: TIME-ZONE VARIABLES FOR EACH TEAM & DATE
#   tz_var[t,d] = time zone of arena where team t plays on date d
# ============================================================================

tz_var = {}
for t in teams:
    for d in team_dates[t]:
        tz_var[(t, d)] = m.addVar(lb=0.0, ub=3.0, vtype=GRB.CONTINUOUS,
                                  name=f"tz_{t}_{d}")

# Link tz_var with x:
for t in teams:
    for d in team_dates[t]:
        # If t is HOME on date d, arena is t's home: fixed timezone
        if d in home_dates.get(t, []):
            m.addConstr(
                tz_var[(t, d)] == tz_map[t],
                name=f"tz_home_{t}_{d}"
            )
        # If t is AWAY on date d, arena is opponent's home
        if d in away_dates.get(t, []):
            # Exactly one i with x[i,d,t] = 1, so this sum is the correct tz
            m.addConstr(
                tz_var[(t, d)] ==
                gp.quicksum(
                    tz_map[i] * x[i, d, t]
                    for i in teams
                    if i != t and (i, d, t) in x
                ),
                name=f"tz_away_{t}_{d}"
            )

# ============================================================================
# PART 5: TRIPLE GAME CONSTRAINT
#   For each team t and each triple of consecutive games (d1,d2,d3):
#     |tz(t,d2) - tz(t,d1)| + |tz(t,d3) - tz(t,d2)| <= 3
# ============================================================================
for t in teams:
    dates_t = team_dates[t]
    for k in range(len(dates_t) - 2):
        d1, d2, d3 = dates_t[k], dates_t[k + 1], dates_t[k + 2]

        diff1 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name=f"diff1_{t}_{k}")
        diff2 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name=f"diff2_{t}_{k}")

        # diff1 >= |tz(d2) - tz(d1)|
        m.addConstr(diff1 >= tz_var[(t, d2)] - tz_var[(t, d1)],
                    name=f"diff1_pos_{t}_{k}")
        m.addConstr(diff1 >= tz_var[(t, d1)] - tz_var[(t, d2)],
                    name=f"diff1_neg_{t}_{k}")

        # diff2 >= |tz(d3) - tz(d2)|
        m.addConstr(diff2 >= tz_var[(t, d3)] - tz_var[(t, d2)],
                    name=f"diff2_pos_{t}_{k}")
        m.addConstr(diff2 >= tz_var[(t, d2)] - tz_var[(t, d3)],
                    name=f"diff2_neg_{t}_{k}")

        # Sum of absolute differences <= 3 (i.e., < 4)
        m.addConstr(diff1 + diff2 <= 3,
                    name=f"tz_trip_{t}_{k}")

# ============================================================================
# PART 6: OBJECTIVE (FEASIBILITY PROBLEM, OBJECTIVE = 0)
# ============================================================================
m.setObjective(0, GRB.MINIMIZE)

# Use solution pool to get multiple feasible schedules
m.Params.PoolSearchMode = 2  # find alternative optimal solutions
m.Params.PoolSolutions = 100  # you can increase this if needed

m.optimize()

# ============================================================================
# PART 7: OUTPUT SOLUTIONS TO CSV
#   Each solution corresponds to a full schedule.
#   We write them as results/schedule_0.csv, schedule_1.csv, ...
# ============================================================================

if m.SolCount == 0:
    print("No feasible schedule exists satisfying the time-zone triple constraint.")
else:
    # Ensure results directory exists
    os.makedirs("results", exist_ok=True)

    print(f"Found {m.SolCount} feasible schedules (solution pool).")
    for sol_num in range(m.SolCount):
        m.Params.SolutionNumber = sol_num

        # Copy original schedule as template
        df_sol = df.copy()

        # For each game slot (row: Date & Home), pick the Visitor from x
        for idx, row in df_sol.iterrows():
            date = row["Date"]
            home = row["Home"]

            # Find which visitor j is chosen in this solution
            chosen_visitor = None
            for j in teams:
                if home == j:
                    continue
                key = (home, date, j)
                if key in x and x[key].Xn > 0.5:  # Xn = value in solution sol_num
                    chosen_visitor = j
                    break

            if chosen_visitor is None:
                # This should not happen if model is correct and constraints enforced
                raise RuntimeError(f"No visitor found for game slot {home} at {date} in solution {sol_num}")

            df_sol.at[idx, "Visitor"] = chosen_visitor

        out_path = os.path.join("results", f"schedule_{sol_num}.csv")
        # Drop helper parsed column before saving
        df_sol_no_parsed = df_sol.drop(columns=["Date_parsed"])
        df_sol_no_parsed.to_csv(out_path, index=False)
        print(f"Saved feasible schedule #{sol_num} to {out_path}")


Set parameter PoolSearchMode to value 2
Set parameter PoolSolutions to value 100
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
PoolSolutions  100
PoolSearchMode  2

Optimize a model with 2112 rows, 1728 columns and 8400 nonzeros
Model fingerprint: 0xe3d9615a
Variable types: 704 continuous, 1024 integer (1024 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 3e+00]
  RHS range        [1e+00, 3e+00]
Presolve removed 996 rows and 718 columns
Presolve time: 0.00s

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 10 available processors)

Solution count 0

Model is infeasible
Best objective -, best bound -, gap -
No feasible schedule exists satisfying the time-zone triple constraint.
